In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
LEAGUES = ["premier_league", "la_liga", "serie_a", "bundesliga", "ligue_1"]
SEASONS = ["2020", "2021", "2022", "2023", "2024"]
RAW_DIR = Path("../data/raw/shots")

In [3]:
shots_list   = []
rosters_list = []

for league in LEAGUES:
    for season in SEASONS:
        folder = RAW_DIR / league / season
        if not folder.exists():
            print(f"Missing: {folder}")
            continue

        files = list(folder.glob("*.json"))
        print(f"{league} {season}: {len(files)} files")

        for path in files:                          # ← indented inside season loop
            with open(path) as f:
                data = json.load(f)

            match_id = int(path.stem)

            for side in ["h", "a"]:
                for shot in data.get("shots", {}).get(side, []):
                    shot["league"] = league
                    shot["year"]   = season
                    shots_list.append(shot)

            for side in ["h", "a"]:
                roster_side = data.get("rosters", {}).get(side, [])
                if isinstance(roster_side, dict):
                    roster_side = roster_side.values()
                for player in roster_side:
                    player["league"]   = league
                    player["year"]     = season
                    player["match_id"] = match_id
                    player["h_a"]      = side
                    rosters_list.append(player)

print(f"\nTotal shots:   {len(shots_list)}")
print(f"Total rosters: {len(rosters_list)}")

premier_league 2020: 380 files
premier_league 2021: 380 files
premier_league 2022: 380 files
premier_league 2023: 380 files
premier_league 2024: 380 files
la_liga 2020: 380 files
la_liga 2021: 380 files
la_liga 2022: 380 files
la_liga 2023: 380 files
la_liga 2024: 380 files
serie_a 2020: 380 files
serie_a 2021: 380 files
serie_a 2022: 380 files
serie_a 2023: 380 files
serie_a 2024: 380 files
bundesliga 2020: 306 files
bundesliga 2021: 306 files
bundesliga 2022: 306 files
bundesliga 2023: 306 files
bundesliga 2024: 306 files
ligue_1 2020: 380 files
ligue_1 2021: 380 files
ligue_1 2022: 380 files
ligue_1 2023: 306 files
ligue_1 2024: 306 files

Total shots:   224676
Total rosters: 273825


In [4]:
shots_df = pd.DataFrame(shots_list)

shots_df['match_id']   = shots_df['match_id'].astype(int)
shots_df['player_id']  = shots_df['player_id'].astype(int)
shots_df['minute']     = shots_df['minute'].astype(int)
shots_df['X']          = shots_df['X'].astype(float)
shots_df['Y']          = shots_df['Y'].astype(float)
shots_df['xG']         = shots_df['xG'].astype(float)
shots_df['h_goals']    = shots_df['h_goals'].astype(int)
shots_df['a_goals']    = shots_df['a_goals'].astype(int)
shots_df['season']     = shots_df['season'].astype(int)
shots_df['date']       = pd.to_datetime(shots_df['date'])

print(shots_df.dtypes)
print(shots_df.shape)
shots_df.head()

id                            str
minute                      int64
result                        str
X                         float64
Y                         float64
xG                        float64
player                        str
h_a                           str
player_id                   int64
situation                     str
season                      int64
shotType                      str
match_id                    int64
h_team                        str
a_team                        str
h_goals                     int64
a_goals                     int64
date               datetime64[us]
player_assisted               str
lastAction                    str
league                        str
year                          str
dtype: object
(224676, 22)


,id,minute,result,X,Y,xG,player,h_a,player_id,situation,...,match_id,h_team,a_team,h_goals,a_goals,date,player_assisted,lastAction,league,year
0,388074,5,MissedShots,0.753,0.491,0.021301,Raphinha,h,8026,OpenPlay,...,14518,Leeds,Arsenal,0,0,2020-11-22 16:30:00,Mateusz Klich,Pass,premier_league,2020
1,388076,11,SavedShot,0.928,0.567,0.452016,Patrick Bamford,h,822,OpenPlay,...,14518,Leeds,Arsenal,0,0,2020-11-22 16:30:00,Ezgjan Alioski,Pass,premier_league,2020
2,388078,18,BlockedShot,0.767,0.634,0.019301,Stuart Dallas,h,8718,OpenPlay,...,14518,Leeds,Arsenal,0,0,2020-11-22 16:30:00,NaN,None,premier_league,2020
3,388079,18,MissedShots,0.741,0.605,0.015293,Jack Harrison,h,8720,OpenPlay,...,14518,Leeds,Arsenal,0,0,2020-11-22 16:30:00,NaN,None,premier_league,2020
4,388082,26,MissedShots,0.763,0.593,0.022445,Raphinha,h,8026,OpenPlay,...,14518,Leeds,Arsenal,0,0,2020-11-22 16:30:00,Ezgjan Alioski,Pass,premier_league,2020


In [5]:
rosters_df = pd.DataFrame(rosters_list)

# cast types
rosters_df['player_id']    = rosters_df['player_id'].astype(int)
rosters_df['team_id']      = rosters_df['team_id'].astype(int)
rosters_df['goals']        = rosters_df['goals'].astype(int)
rosters_df['own_goals']    = rosters_df['own_goals'].astype(int)
rosters_df['shots']        = rosters_df['shots'].astype(int)
rosters_df['xG']           = rosters_df['xG'].astype(float)
rosters_df['xA']           = rosters_df['xA'].astype(float)
rosters_df['xGChain']      = rosters_df['xGChain'].astype(float)
rosters_df['xGBuildup']    = rosters_df['xGBuildup'].astype(float)
rosters_df['time']         = rosters_df['time'].astype(int)
rosters_df['yellow_card']  = rosters_df['yellow_card'].astype(int)
rosters_df['red_card']     = rosters_df['red_card'].astype(int)
rosters_df['roster_in']    = rosters_df['roster_in'].astype(int)
rosters_df['roster_out']   = rosters_df['roster_out'].astype(int)
rosters_df['key_passes']   = rosters_df['key_passes'].astype(int)
rosters_df['assists']      = rosters_df['assists'].astype(int)

print(rosters_df.dtypes)
print(rosters_df.shape)
rosters_df.head()

id                   str
goals              int64
own_goals          int64
shots              int64
xG               float64
time               int64
player_id          int64
team_id            int64
position             str
player               str
h_a                  str
yellow_card        int64
red_card           int64
roster_in          int64
roster_out         int64
key_passes         int64
assists            int64
xA               float64
xGChain          float64
xGBuildup        float64
positionOrder        str
league               str
year                 str
match_id           int64
dtype: object
(273825, 24)


,id,goals,own_goals,shots,xG,time,player_id,team_id,position,player,...,roster_out,key_passes,assists,xA,xGChain,xGBuildup,positionOrder,league,year,match_id
0,428820,0,0,0,0.000000,90,8715,245,GK,Illan Meslier,...,0,0,0,0.000000,0.017848,0.017848,1,premier_league,2020,14518
1,428821,0,0,1,0.060715,70,8716,245,DR,Luke Ayling,...,0,0,0,0.000000,0.098097,0.098097,2,premier_league,2020,14518
2,428823,0,0,1,0.063235,90,8816,245,DC,Liam Cooper,...,0,0,0,0.000000,0.515682,0.515682,3,premier_league,2020,14518
3,428822,0,0,1,0.036030,90,6273,245,DC,Robin Koch,...,0,3,0,0.116626,0.517942,0.419164,3,premier_league,2020,14518
4,428824,0,0,1,0.026770,90,8722,245,DL,Ezgjan Alioski,...,0,2,0,0.474462,1.071970,1.022750,4,premier_league,2020,14518


In [7]:
def assign_phase(minute):
    if minute <= 25:
        return 'Q1'
    elif minute <= 45:
        return 'Q2'
    elif minute <= 70:
        return 'Q3'
    elif minute <= 90:
        return 'Q4'
    else:
        return 'AT'  # added time

shots_df['phase'] = shots_df['minute'].apply(assign_phase)

shots_df['goal_diff'] = np.where(
    shots_df['h_a'] == 'h',
    shots_df['h_goals'] - shots_df['a_goals'],
    shots_df['a_goals'] - shots_df['h_goals']
)

shots_df['game_state'] = pd.cut(
    shots_df['goal_diff'],
    bins=[-np.inf, -1, 0, 1, np.inf],
    labels=['losing','drawing','winning_by_1','winning_by_2+']
)

shots_df['cross_side'] = np.where(shots_df['Y'] < 0.5, 'left', 'right')

shots_df['is_cross'] = shots_df['lastAction'] == 'Cross'

shots_df['shot_zone'] = pd.cut(
    shots_df['X'],
    bins=[0, 0.7, 0.85, 1.0],
    labels=['outside_box','edge_of_box','six_yard_box'],
    include_lowest=True
)

print(shots_df[['phase','goal_diff','game_state','cross_side','shot_zone']].head(10))
print(shots_df['phase'].value_counts())
print(shots_df['game_state'].value_counts())

  phase  goal_diff game_state cross_side     shot_zone
0    Q1          0    drawing       left   edge_of_box
1    Q1          0    drawing      right  six_yard_box
2    Q1          0    drawing      right   edge_of_box
3    Q1          0    drawing      right   edge_of_box
4    Q2          0    drawing      right   edge_of_box
5    Q2          0    drawing       left  six_yard_box
6    Q2          0    drawing      right   edge_of_box
7    Q2          0    drawing      right  six_yard_box
8    Q2          0    drawing       left  six_yard_box
9    Q2          0    drawing      right   edge_of_box
phase
Q3    65397
Q1    53722
Q4    47518
Q2    46858
AT    11181
Name: count, dtype: int64
game_state
losing           75137
drawing          56239
winning_by_2+    50081
winning_by_1     43219
Name: count, dtype: int64


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)

shots_df.to_csv("../data/processed/shots.csv", index=False)
rosters_df.to_csv("../data/processed/rosters.csv", index=False)

print("Saved successfully")
print("shots_df:  ", shots_df.shape)
print("rosters_df:", rosters_df.shape)

Saved successfully
shots_df:   (224676, 28)
rosters_df: (273825, 24)


In [10]:
print(rosters_df.columns.tolist())
print(rosters_df['match_id'].isna().sum())
print(rosters_df[['match_id','h_a','player','team_id','league','year']].head())


['id', 'goals', 'own_goals', 'shots', 'xG', 'time', 'player_id', 'team_id', 'position', 'player', 'h_a', 'yellow_card', 'red_card', 'roster_in', 'roster_out', 'key_passes', 'assists', 'xA', 'xGChain', 'xGBuildup', 'positionOrder', 'league', 'year', 'match_id']
0
   match_id h_a          player  team_id          league  year
0     14518   h   Illan Meslier      245  premier_league  2020
1     14518   h     Luke Ayling      245  premier_league  2020
2     14518   h     Liam Cooper      245  premier_league  2020
3     14518   h      Robin Koch      245  premier_league  2020
4     14518   h  Ezgjan Alioski      245  premier_league  2020
